# Audio De-identification

**Thursday Masking Day · Tilburg Multiscale Summerschool 2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_ORG/TilburgMultiscaleSummerschool2026/blob/main/thursday-masking/notebooks/04_audio_deidentification.ipynb)

---

Voice is **biometric data** — arguably more identifying than a face.  
Even after you mask the video, an unaltered audio track can re-identify participants.

This notebook walks through three practical audio de-identification strategies,  
from quick-and-easy to robust, and shows how to evaluate the trade-off between  
privacy and the preservation of speech features you care about.

### Methods covered

| # | Method | Privacy | Naturalness | Speed |
|---|--------|---------|-------------|-------|
| 1 | **Pitch shifting** | Low–Medium | High | Fast |
| 2 | **Noise injection + formant scrambling** | Medium | Medium | Fast |
| 3 | **Speech → Text → Speech** | High | Lower | Slow |

### Connection to Thursday

This session connects to:  
- **Tuesday** acoustic analysis — de-id changes prosodic features you may want to measure  
- **Wednesday** pose estimation — audio + video masking decisions interact  
- **Notebook 05** archiving — audio is archived separately from video (different access tier)


## Setup

Run this cell first. On Colab it installs everything you need automatically.

In [ ]:
# Install dependencies (Colab & local)
import sys
!{sys.executable} -m pip install -q \
    librosa soundfile pydub numpy scipy matplotlib ipython

# ffmpeg is required by pydub — check it's available
import shutil
if not shutil.which("ffmpeg"):
    print("ffmpeg not found. On Colab: !apt-get install -y -q ffmpeg")
    print("On Windows: download from https://ffmpeg.org/download.html")
else:
    print("ffmpeg OK")

import librosa
import librosa.display
import soundfile as sf
import numpy as np
import scipy.signal
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from IPython.display import Audio, display

plt.rcParams.update({
    'figure.facecolor': '#161616',
    'axes.facecolor':   '#262626',
    'axes.edgecolor':   '#525252',
    'axes.labelcolor':  '#c6c6c6',
    'xtick.color':      '#6f6f6f',
    'ytick.color':      '#6f6f6f',
    'text.color':       '#f4f4f4',
    'grid.color':       '#393939',
    'grid.linestyle':   '--',
})

OUT_DIR = Path("audio_deid_outputs")
OUT_DIR.mkdir(exist_ok=True)
print("Setup complete.")

## Load your audio

Use one of three options below depending on where you're running this.

> **On Colab:** upload a `.wav` or `.mp4` file using the Files panel on the left,  
> or mount Google Drive and point `AUDIO_PATH` to your file.

In [ ]:
# ── Option A: load a local file ────────────────────────────────
# AUDIO_PATH = Path("../data/trimmed/group_07/group_07__cam_p1.mp4")

# ── Option B: generate a synthetic voice sample for demo ───────
def make_demo_audio(duration=10, sr=16000) -> tuple:
    """Generate a simple voiced signal for demo purposes."""
    t = np.linspace(0, duration, int(sr * duration))
    # Simulate a voiced tone with harmonics and slight pitch variation
    f0 = 150 + 20 * np.sin(2 * np.pi * 0.5 * t)  # slowly varying f0
    signal = sum(
        (1 / k) * np.sin(2 * np.pi * k * f0 * t)
        for k in range(1, 10)
    )
    signal = signal / np.max(np.abs(signal)) * 0.6
    # Add breath noise
    signal += np.random.randn(len(t)) * 0.05
    return signal.astype(np.float32), sr

# ── Option C: extract audio from an MP4 ────────────────────────
def extract_audio_from_video(mp4_path: Path, sr=16000) -> tuple:
    import subprocess, tempfile
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        subprocess.run([
            "ffmpeg", "-y", "-i", str(mp4_path),
            "-ac", "1", "-ar", str(sr), tmp.name
        ], capture_output=True, check=True)
        y, _ = librosa.load(tmp.name, sr=sr)
    return y, sr

# --- CHOOSE YOUR SOURCE ---
AUDIO_PATH = None   # set to Path(...) to use a real file

if AUDIO_PATH and AUDIO_PATH.suffix == ".mp4":
    y_orig, SR = extract_audio_from_video(AUDIO_PATH)
elif AUDIO_PATH:
    y_orig, SR = librosa.load(AUDIO_PATH, sr=None, mono=True)
else:
    print("No file specified — using synthetic demo audio.")
    y_orig, SR = make_demo_audio()

print(f"Audio loaded: {len(y_orig)/SR:.1f}s · {SR} Hz")
display(Audio(y_orig, rate=SR))

## Visualise the original audio

Before anonymising, let's see what we're working with —  
waveform, spectrogram, and a rough pitch (F0) estimate.

In [ ]:
def plot_audio(y, sr, title="Audio", ax=None):
    fig = None
    if ax is None:
        fig, axes = plt.subplots(2, 1, figsize=(12, 5),
                                  gridspec_kw={"height_ratios": [1, 2]})
    else:
        axes = ax

    times = np.linspace(0, len(y) / sr, len(y))
    axes[0].plot(times, y, color="#78a9ff", linewidth=0.5)
    axes[0].set_ylabel("Amplitude")
    axes[0].set_title(title, color="#f4f4f4", fontsize=11)
    axes[0].set_xlim(0, len(y) / sr)

    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis="time", y_axis="hz",
                              ax=axes[1], cmap="magma")
    axes[1].set_ylabel("Frequency (Hz)")

    if fig:
        plt.tight_layout()
        plt.show()

plot_audio(y_orig, SR, title="Original")

---
## Method 1: Pitch Shifting

**What it does:** Raises or lowers the fundamental frequency (F0) of the voice  
while keeping duration constant.

**Privacy level:** Low–Medium — voice timbre and rhythm are preserved.  
A determined listener or voice classifier may still identify the speaker.

**Best for:** Quick anonymisation of speech where prosody still matters  
(e.g. emotional tone, turn-taking patterns).

**What it destroys:** Exact F0 values — don't use for pitch analysis downstream.

In [ ]:
def pitch_shift(y, sr, n_steps: float) -> np.ndarray:
    """
    Shift pitch by n_steps semitones.
    Positive = higher pitch, negative = lower pitch.
    Typical range: -6 to +6 semitones.
    """
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)

shifts = {
    "Higher (+4 st)":  pitch_shift(y_orig, SR, n_steps=+4),
    "Lower  (−4 st)": pitch_shift(y_orig, SR, n_steps=-4),
    "Extreme (+8 st)": pitch_shift(y_orig, SR, n_steps=+8),
}

for label, y_shifted in shifts.items():
    out = OUT_DIR / f"pitch_{label[:5].strip()}.wav"
    sf.write(out, y_shifted, SR)
    print(f"\n{label}")
    display(Audio(y_shifted, rate=SR))

# Compare waveforms
fig, axes = plt.subplots(2, 2, figsize=(14, 5))
plot_audio(y_orig, SR, title="Original", ax=axes[0])
for i, (label, y_s) in enumerate(shifts.items(), 1):
    plot_audio(y_s, SR, title=label, ax=axes[i])
plt.tight_layout(); plt.show()

---
## Method 2: Noise Injection + Formant Scrambling

**What it does:** Adds calibrated noise to mask fine spectral detail,  
then applies a mild time-stretching + re-sampling trick to shift formants  
without obviously changing perceived pitch.

**Privacy level:** Medium — harder to classify with automatic speaker ID systems.  
Speech remains intelligible if noise level is kept low.

**Best for:** Datasets where you need the audio to remain listenable for  
qualitative annotation, but automatic speaker identification must be prevented.

In [ ]:
def add_noise(y: np.ndarray, snr_db: float = 20) -> np.ndarray:
    """Add white noise at a specified signal-to-noise ratio (dB)."""
    signal_power = np.mean(y ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.random.randn(len(y)) * np.sqrt(noise_power)
    return (y + noise).astype(np.float32)

def formant_scramble(y: np.ndarray, sr: int, rate: float = 0.92) -> np.ndarray:
    """
    Approximate formant shift via time-stretch + resample.
    rate < 1 raises formants, rate > 1 lowers them.
    """
    y_stretched = librosa.effects.time_stretch(y, rate=rate)
    target_len  = len(y)
    y_resampled = scipy.signal.resample(y_stretched, target_len)
    return y_resampled.astype(np.float32)

# Try different noise levels
noise_levels = {"High privacy (SNR 10 dB)": 10,
                "Balanced   (SNR 20 dB)": 20,
                "Subtle     (SNR 35 dB)": 35}

for label, snr in noise_levels.items():
    y_noisy = add_noise(y_orig, snr_db=snr)
    y_deid  = formant_scramble(y_noisy, SR, rate=0.93)
    out = OUT_DIR / f"noise_snr{snr}.wav"
    sf.write(out, y_deid, SR)
    print(f"\n{label}")
    display(Audio(y_deid, rate=SR))

print("\nTip: SNR 20 dB is often a good balance for behavioral research data.")

---
## Method 3: Speech → Text → Speech

**What it does:** Transcribes speech with Whisper, then re-synthesises it  
with a different (synthetic) voice using a text-to-speech (TTS) model.

**Privacy level:** High — the original voice signal is completely replaced.  
Speaker identity is not recoverable from the audio.

**What it destroys:**
- Original voice identity (intended)
- Prosody, emotion, speaking rate nuances
- Non-speech sounds (laughter, hesitation, breath)
- Transcription errors compound

**Best for:** Archives where audio access is needed for content only —  
e.g. keyword search, thematic analysis, not prosodic analysis.

In [ ]:
# ── Step 3a: Install Whisper (ASR) and a TTS engine ────────────
!{sys.executable} -m pip install -q openai-whisper
# For TTS we use pyttsx3 (offline, no API key)
!{sys.executable} -m pip install -q pyttsx3

print("Whisper and TTS installed.")

In [ ]:
import whisper
import tempfile, os

def transcribe(y: np.ndarray, sr: int, model_size="tiny") -> str:
    """
    Transcribe audio using OpenAI Whisper.
    model_size: 'tiny' (fast), 'base', 'small', 'medium', 'large'
    """
    model = whisper.load_model(model_size)
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
        sf.write(tmp.name, y, sr)
        result = model.transcribe(tmp.name, language="en")
        os.unlink(tmp.name)
    return result["text"]

print("Transcribing original audio (Whisper tiny — fast) ...")
transcript = transcribe(y_orig, SR, model_size="tiny")
print(f"Transcript:\n  {transcript!r}")

In [ ]:
import pyttsx3, threading

def text_to_speech(text: str, out_path: Path, rate: int = 160) -> Path:
    """Synthesise text to WAV using pyttsx3 (offline, system TTS)."""
    engine = pyttsx3.init()
    engine.setProperty("rate", rate)
    # Pick a different voice if multiple are available
    voices = engine.getProperty("voices")
    if len(voices) > 1:
        engine.setProperty("voice", voices[1].id)  # alternate voice
    engine.save_to_file(text, str(out_path))
    engine.runAndWait()
    return out_path

tts_out = OUT_DIR / "s2t2s_output.wav"
text_to_speech(transcript, tts_out)

y_tts, sr_tts = librosa.load(tts_out, sr=None)
print("Synthesised speech:")
display(Audio(y_tts, rate=sr_tts))

print("\n Original:")
display(Audio(y_orig, rate=SR))

---
## Evaluation: what did de-identification destroy?

We compare three acoustic features that behavioral researchers often measure:  
- **Pitch (F0)** — fundamental frequency, important for prosody and emotion  
- **Spectral centroid** — brightness / quality of sound  
- **RMS energy** — loudness over time (rhythm, emphasis)

Each method preserves these features to different degrees.

In [ ]:
def extract_features(y, sr, label):
    f0, voiced, _ = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
    centroid       = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    rms            = librosa.feature.rms(y=y)[0]
    return {
        "label":             label,
        "f0_mean":           float(np.nanmean(f0)),
        "f0_std":            float(np.nanstd(f0)),
        "centroid_mean":     float(np.mean(centroid)),
        "rms_mean":          float(np.mean(rms)),
    }

y_pitch  = pitch_shift(y_orig, SR, n_steps=+4)
y_noisy  = formant_scramble(add_noise(y_orig, snr_db=20), SR)

results = [
    extract_features(y_orig,  SR,     "Original"),
    extract_features(y_pitch, SR,     "Pitch +4st"),
    extract_features(y_noisy, SR,     "Noise+Formant"),
]

# Simple table
print(f"{'Method':<20} {'F0 mean':>10} {'F0 std':>8} {'Centroid':>10} {'RMS':>8}")
print("-" * 60)
for r in results:
    print(f"{r['label']:<20} {r['f0_mean']:>10.1f} "
          f"{r['f0_std']:>8.1f} {r['centroid_mean']:>10.0f} {r['rms_mean']:>8.4f}")

In [ ]:
# Visualise F0 contours side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 3), sharey=True)

for ax, (y, label) in zip(axes, [
        (y_orig,  "Original"),
        (y_pitch, "Pitch +4st"),
        (y_noisy, "Noise+Formant")]):
    f0, voiced, _ = librosa.pyin(y, fmin=50, fmax=500, sr=SR)
    t = librosa.times_like(f0, sr=SR)
    ax.plot(t, f0, color="#78a9ff", linewidth=1)
    ax.set_title(label)
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("F0 (Hz)" if ax == axes[0] else "")
    ax.set_ylim(0, 500)

plt.suptitle("F0 contours — de-identification comparison",
             color="#f4f4f4", y=1.02)
plt.tight_layout()
plt.show()

---
## Bonus: Speaker diarization before de-identification

When you have multiple speakers (as in the Envision triadic sessions),  
you should **diarise first** (identify who speaks when) and apply per-speaker  
anonymisation so each participant gets a distinct synthetic voice.

The best tool for this is **pyannote.audio**, but it requires a HuggingFace token.  
The cell below shows the pattern — fill in your token to run it.

In [ ]:
# Install pyannote (large download — ~2 GB models)
# !{sys.executable} -m pip install -q pyannote.audio

HF_TOKEN = None   # replace with your HuggingFace token

def diarise_and_deid(audio_path: Path, token: str, out_dir: Path):
    """
    1. Diarise audio into speaker segments
    2. Apply pitch shift per speaker (different n_steps per ID)
    3. Reassemble into a single anonymised track
    """
    from pyannote.audio import Pipeline
    import pydub

    pipeline = Pipeline.from_pretrained(
        "pyannote/speaker-diarization-3.1",
        use_auth_token=token
    )
    diarization = pipeline(str(audio_path))

    y, sr = librosa.load(audio_path, sr=16000, mono=True)
    result = np.zeros_like(y)

    speaker_shifts = {}   # assign a unique pitch shift per speaker
    shift_pool     = [+3, -4, +6, -3, +5, -6]

    for turn, _, speaker in diarization.itertracks(yield_label=True):
        if speaker not in speaker_shifts:
            speaker_shifts[speaker] = shift_pool[len(speaker_shifts) % len(shift_pool)]
        n_steps = speaker_shifts[speaker]

        start = int(turn.start * sr)
        end   = int(turn.end   * sr)
        segment = y[start:end]
        shifted = pitch_shift(segment, sr, n_steps)
        result[start:end] += shifted

    out_path = out_dir / "diarised_deid.wav"
    sf.write(out_path, result, sr)
    print(f"Speakers: {speaker_shifts}")
    print(f"Output:   {out_path}")
    return out_path, speaker_shifts

if HF_TOKEN:
    diarise_and_deid(AUDIO_PATH or OUT_DIR / "noise_snr20.wav",
                     token=HF_TOKEN, out_dir=OUT_DIR)
else:
    print("Set HF_TOKEN above to run speaker diarization.")
    print("Register at: https://hf.co and accept the pyannote terms.")

---
## Summary: choosing a method

| Your need | Recommended method |
|-----------|--------------------|
| Quick turnaround, prosody matters | Pitch shifting (±4 semitones) |
| GDPR compliance, audio still needed | Noise + formant (SNR 20 dB) |
| Maximum privacy, content only | Speech → Text → Speech |
| Multi-speaker session | Diarise first, then per-speaker pitch shift |
| Audio not needed at all | Strip from video (`ffmpeg -an`) — see notebook 05 |

### Connection to the archive (notebook 05)

- De-identified audio → **embargoed tier** (can share with collaborators)
- Original audio → **restricted tier** (stored only on SYNAPSIS, not shared)
- Video-only (silent masked MP4) → **open tier** (DANS Data Station SSH)

**Reference:**  
Owoyele et al. (2026). MaskingOPS. *Behavior Research Methods*. (under review)  
VoicePrivacy Challenge: [voiceprivacychallenge.univ-avignon.fr](https://voiceprivacychallenge.univ-avignon.fr)
